# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)


# Save the correct dataset to a new CSV file
df = px.data.gapminder()

df.to_csv(
    '/content/drive/MyDrive/dataviz/gapminder_correct.csv',
    index=False
)
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [7]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [8]:
# Task 1
# YOUR CODE HERE




import numpy as np


europe = df[
    (df['continent'] == 'Europe') &
    (df['year'].isin([2002, 2007]))
].copy()


europe['year'] = europe['year'].astype(str)

# scatter plot
fig = px.scatter(
    europe,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map={
        '2002': 'royalblue',
        '2007': 'darkorange'
    },
    hover_name='country',
    log_x=True,
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'year': 'Year'
    },
    title='Most European countries improved in GDP and life expectancy from 2002 to 2007',
    template='plotly_white'
)


for country, group in europe.groupby('country'):
    old = group[group['year'] == '2002'].iloc[0]
    new = group[group['year'] == '2007'].iloc[0]

    fig.add_annotation(
        x=np.log10(new['gdpPercap']),
        y=new['lifeExp'],
        ax=np.log10(old['gdpPercap']),
        ay=old['lifeExp'],
        xref='x',
        yref='y',
        axref='x',
        ayref='y',
        text='',
        showarrow=True,
        arrowhead=2,
        arrowcolor='grey',
        arrowwidth=1
    )

fig.update_traces(marker=dict(size=9, opacity=0.85))
fig.update_layout(height=550)

fig.show()



## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [10]:
# Task 2
# YOUR CODE HERE



data_2007 = df[df['year'] == 2007].copy()


highlight = 'Europe'

colors = {
    c: ('royalblue' if c == highlight else 'lightgrey')
    for c in data_2007['continent'].unique()
}

#  bubble chart
fig = px.scatter(
    data_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    color_discrete_map=colors,
    category_orders={
        'continent': [
            'Africa', 'Americas', 'Asia',
            'Oceania', 'Europe'
        ]
    },
    hover_name='country',
    log_x=True,
    size_max=55,
    opacity=0.7,
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population'
    },
    title='European countries generally had high GDP and life expectancy in 2007',
    template='plotly_white'
)


fig.add_annotation(
    x=0.5,
    y=0.95,
    xref='paper',
    yref='paper',
    text='European countries mostly have higher<br>GDP and life expectancy.',
    showarrow=False,
    font=dict(size=12, color='royalblue'),
    bgcolor='white',
    bordercolor='royalblue',
    borderwidth=1
)

fig.update_layout(
    height=550,
    showlegend=False
)

fig.show()
